# ConverSight AI: Data Ingestion & Pipeline Engineering
### Transcript Intelligence Platform - Data Pipeline & Schema Design

Welcome to the **Data Ingestion & Pipeline Engineering** notebook. This notebook contains the step-by-step implementation of the data engineering pipeline for the ConverSight AI platform.

#### Ingestion Pipeline Goals:
1. **Analytical Schema Setup**: Initialize DuckDB tables for meetings, transcripts, action items, topics, and knowledge graphs.
2. **Validation & Corruption Gate**: Filter out incomplete or malformed meetings from the raw dataset.
3. **B2B Account Resolution Heuristics**: Resolve raw meetings to enterprise customer accounts using title matching and email domain extraction.
4. **Speaker & Role Attribution**: Attribute contextual speaker roles (Support Agent, Engineer, Account Manager, Customer) based on domains.
5. **Action Item NLP Extraction**: Parse owner name and run regex-based deadline parsing on raw action item strings.
6. **Entity Mentions Harvesting**: Track discussions of competitors (Okta, Duo, Ping, Entra) and Aegis products.
7. **Idempotent Graph Ingestion**: Construct and upload unique nodes and edges to build a connected network without duplicates.

## 1. Setup & Schema Definition

We start by connecting to our DuckDB database (isolated testing file `conversight_pipeline_test.db` to prevent locking conflicts with running services) and creating the relational tables representing our unified schema model.

In [ ]:
import os
import json
import re
from datetime import datetime
import pandas as pd
import duckdb

# Set test database path to avoid locking conflicts with live dashboard backend
db_path = "conversight_pipeline_test.db"

conn = duckdb.connect(db_path)
print(f"Database connection established at: {db_path}")

# Enable DuckDB FTS (Full Text Search) extension
try:
    conn.execute("INSTALL fts; LOAD fts;")
    print("DuckDB FTS extension loaded successfully.")
except Exception as e:
    print(f"Warning: FTS extension load failed: {e}")

In [ ]:
# Create analytical relational and graph tables
conn.execute("""
CREATE TABLE IF NOT EXISTS meetings (
    meeting_id VARCHAR PRIMARY KEY,
    title VARCHAR,
    organizer_email VARCHAR,
    host VARCHAR,
    start_time TIMESTAMP,
    end_time TIMESTAMP,
    duration DOUBLE,
    summary VARCHAR,
    overall_sentiment VARCHAR,
    sentiment_score DOUBLE,
    call_type VARCHAR
);

CREATE TABLE IF NOT EXISTS transcript_segments (
    id VARCHAR PRIMARY KEY,
    meeting_id VARCHAR,
    sentence VARCHAR,
    speaker_name VARCHAR,
    speaker_role VARCHAR,
    sentiment_type VARCHAR,
    time DOUBLE,
    end_time DOUBLE,
    average_confidence DOUBLE,
    turn_index INTEGER
);

CREATE TABLE IF NOT EXISTS action_items (
    id VARCHAR PRIMARY KEY,
    meeting_id VARCHAR,
    task VARCHAR,
    owner VARCHAR,
    deadline VARCHAR,
    status VARCHAR
);

CREATE TABLE IF NOT EXISTS topics (
    meeting_id VARCHAR,
    topic VARCHAR,
    PRIMARY KEY (meeting_id, topic)
);

CREATE TABLE IF NOT EXISTS key_moments (
    id VARCHAR PRIMARY KEY,
    meeting_id VARCHAR,
    time DOUBLE,
    text VARCHAR,
    type VARCHAR,
    speaker VARCHAR
);

CREATE TABLE IF NOT EXISTS entities (
    meeting_id VARCHAR,
    entity_name VARCHAR,
    entity_type VARCHAR,
    PRIMARY KEY (meeting_id, entity_name, entity_type)
);

CREATE TABLE IF NOT EXISTS graph_nodes (
    id VARCHAR PRIMARY KEY,
    label VARCHAR,
    type VARCHAR
);

CREATE TABLE IF NOT EXISTS graph_edges (
    source VARCHAR,
    target VARCHAR,
    relation VARCHAR,
    PRIMARY KEY (source, target, relation)
);
""")
print("Relational & Knowledge Graph tables initialized successfully.")

## 2. Ingestion Pipeline Logics & Heuristics

Let's define the helper logics that process raw JSON data into enriched, standardized variables.

In [ ]:
KNOWN_CUSTOMERS = [
    'Steelpoint Manufacturing', 'Vanta Health Systems', 'Crestline Wealth Group', 'Crestline Wealth', 'Coastal Living Co', 
    'Maplewood Goods', 'Helix Data', 'Redwood Clinical', 'Quantum Edge', 'Trailhead Marketplace', 
    'Axiom Labs', 'Keystone Health', 'Northstar Pharma', 'Summit Trust', 'Brightpath Commerce', 
    'Ridgeline Logistics', 'Bridgeport Health', 'Clearwater Medical', 'Blackridge Investments', 
    'Ironworks Corp', 'Harborview Banking', 'Nimbus Platform', 'Cobalt Software', 'Pinnacle Insurance', 
    'Pineridge Systems', 'Silverline Brands', 'Forge Industries', 'Meridian Capital', 'Atlas Precision', 
    'Ironclad Financial', 'Frostbyte AI', 'Stratos Cloud', 'Nova Retail Group'
]

COMPETITORS = ['Okta', 'Duo', 'Ping', 'Entra']
PRODUCTS = ['Aegis Identity', 'Aegis Detect', 'Aegis Comply', 'Aegis Protect', 'Aegis Sentinel', 'Aegis Compliance']

def extract_customer(title, emails):
    # Heuristic 1: Predefined Title matching
    for cust in KNOWN_CUSTOMERS:
        if cust.lower() in title.lower():
            return cust
    
    # Heuristic 2: Email domain resolution (ignoring corporate or standard public consumer emails)
    for email in emails:
        if "@" in email:
            domain = email.split("@")[1].split(".")[0]
            if domain not in ["aegiscloud", "aegis", "gmail", "outlook", "hotmail", "yahoo"]:
                return domain.replace("corp", "Corp").replace("inc", "Inc").title()
                
    return "Unknown Customer"

def clean_action_item(item_str):
    # Normalise action items to split owner name from task text
    parts = item_str.split(":", 1)
    if len(parts) == 2:
        owner = parts[0].strip()
        task = parts[1].strip()
    else:
        owner = "Unassigned"
        task = item_str.strip()
    
    # Regular expression deadline parser
    deadline = "N/A"
    deadline_matches = [
        (r"this week", "End of Week"),
        (r"by friday", "Friday"),
        (r"next week", "Next Week"),
        (r"within (\d+)\s*(business\s*)?days", r"In \1 days"),
        (r"before the (\w+)\s*billing", r"Before \1 billing"),
    ]
    for pattern, label in deadline_matches:
        match = re.search(pattern, task, re.IGNORECASE)
        if match:
            if r"\1" in label:
                deadline = label.replace(r"\1", match.group(1))
            else:
                deadline = label
            break
            
    return owner, task, deadline

def parse_date(date_str):
    if not date_str:
        return None
    try:
        return date_str.replace("Z", "+00:00")
    except Exception:
        return date_str
print("Ingestion logics and heuristics initialized successfully.")

## 3. Bulk Ingestion Engine

This function reads all folders in the target directory, performs validation checks, runs normalizations, and executes idempotent batch writes into DuckDB.

In [ ]:
def ingest_transcripts_pipeline(dataset_dir, connection):
    if not os.path.exists(dataset_dir):
        print(f"Error: Path {dataset_dir} does not exist.")
        return
        
    print(f"Scanning directory: {dataset_dir}...")
    dirs = [d for d in os.listdir(dataset_dir) if os.path.isdir(os.path.join(dataset_dir, d))]
    print(f"Found {len(dirs)} subdirectories.")
    
    # Reset existing records to avoid duplicates
    connection.execute("DELETE FROM meetings;")
    connection.execute("DELETE FROM transcript_segments;")
    connection.execute("DELETE FROM action_items;")
    connection.execute("DELETE FROM topics;")
    connection.execute("DELETE FROM key_moments;")
    connection.execute("DELETE FROM entities;")
    connection.execute("DELETE FROM graph_nodes;")
    connection.execute("DELETE FROM graph_edges;")
    
    nodes_to_insert = {} # id -> (label, type)
    edges_to_insert = set() # (source, target, relation)
    
    # Setup default products & competitors nodes
    for prod in PRODUCTS:
        nodes_to_insert[prod] = (prod, 'Product')
    for comp in COMPETITORS:
        nodes_to_insert[comp] = (comp, 'Competitor')
        
    processed = 0
    errors = []
    required_files = ["summary.json", "meeting-info.json", "speakers.json", "events.json", "speaker-meta.json", "transcript.json"]
    
    for d in dirs:
        dir_path = os.path.join(dataset_dir, d)
        files = os.listdir(dir_path)
        
        # Validation Gate: Corruption check
        is_corrupt = False
        for req in required_files:
            if req not in files:
                errors.append(f"Corruption Warning: Missing {req} in folder {d}")
                is_corrupt = True
                break
        if is_corrupt:
            continue
            
        try:
            with open(os.path.join(dir_path, "meeting-info.json")) as f: meeting_info = json.load(f)
            with open(os.path.join(dir_path, "summary.json")) as f: summary = json.load(f)
            with open(os.path.join(dir_path, "transcript.json")) as f: transcript = json.load(f)
            with open(os.path.join(dir_path, "speakers.json")) as f: speakers = json.load(f)
            with open(os.path.join(dir_path, "speaker-meta.json")) as f: speaker_meta = json.load(f)
            with open(os.path.join(dir_path, "events.json")) as f: events = json.load(f)
        except Exception as e:
            errors.append(f"Syntax/Decode Error in {d}: {e}")
            continue
            
        meeting_id = meeting_info.get("meetingId")
        title = meeting_info.get("title", "Untitled Meeting")
        
        # Call type heuristics
        call_type = "External"
        t_lower = title.lower()
        if "support case #" in t_lower or "support case" in t_lower:
            call_type = "Support"
        elif any(k in t_lower for k in ["sprint planning", "all hands", "team sync", "internal", "soc 2", "audit prep", "planning discussion", "wrap"]):
            call_type = "Internal"
        elif not meeting_info.get("invitees") or len(meeting_info.get("invitees")) <= 1:
            call_type = "Internal"
            
        # Resolve Customer Account
        customer = extract_customer(title, meeting_info.get("allEmails", []))
        
        # Write meeting details
        connection.execute("""
        INSERT INTO meetings (meeting_id, title, organizer_email, host, start_time, end_time, duration, summary, overall_sentiment, sentiment_score, call_type)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            meeting_id, title, meeting_info.get("organizerEmail"), meeting_info.get("host"),
            parse_date(meeting_info.get("startTime")), parse_date(meeting_info.get("endTime")),
            meeting_info.get("duration"), summary.get("summary"), summary.get("overallSentiment"),
            summary.get("sentimentScore"), call_type
        ))
        
        # Construct meeting and customer graph nodes
        nodes_to_insert[meeting_id] = (title, 'Meeting')
        if call_type != "Internal" and customer != "Unknown Customer":
            nodes_to_insert[customer] = (customer, 'Customer')
            edges_to_insert.add((meeting_id, customer, 'AFFECTS'))
            
        # Normalize & load action items
        for idx, ai in enumerate(summary.get("actionItems", [])):
            owner, task, deadline = clean_action_item(ai)
            ai_id = f"{meeting_id}_ai_{idx}"
            connection.execute("""
            INSERT INTO action_items (id, meeting_id, task, owner, deadline, status)
            VALUES (?, ?, ?, ?, ?, 'Open')
            """, (ai_id, meeting_id, task, owner, deadline))
            
            nodes_to_insert[ai_id] = (task[:40] + "...", 'ActionItem')
            edges_to_insert.add((meeting_id, ai_id, 'HAS_ACTION_ITEM'))
            if owner != "Unassigned":
                nodes_to_insert[owner] = (owner, 'Speaker')
                edges_to_insert.add((owner, ai_id, 'OWNS'))
                
        # Load topics discussed
        for topic in summary.get("topics", []):
            connection.execute("""
            INSERT INTO topics (meeting_id, topic) VALUES (?, ?) ON CONFLICT DO NOTHING
            """, (meeting_id, topic))
            nodes_to_insert[topic] = (topic, 'Topic')
            edges_to_insert.add((meeting_id, topic, 'DISCUSSES_TOPIC'))
            
        # Load key moments
        for idx, km in enumerate(summary.get("keyMoments", [])):
            connection.execute("""
            INSERT INTO key_moments (id, meeting_id, time, text, type, speaker)
            VALUES (?, ?, ?, ?, ?, ?)
            """, (f"{meeting_id}_km_{idx}", meeting_id, km.get("time"), km.get("text"), km.get("type"), km.get("speaker")))
            
        # Extract entity mentions from transcripts
        mentioned_competitors = set()
        mentioned_products = set()
        
        for idx, turn in enumerate(transcript.get("data", [])):
            sentence = turn.get("sentence", "")
            speaker = turn.get("speaker_name", "Unknown")
            
            # Contextual Speaker Role attribution logic
            role = "Customer"
            s_lower = speaker.lower()
            h_domain = ""
            host_email = meeting_info.get("host", "")
            if "@" in host_email:
                h_domain = host_email.split("@")[1]
            
            speaker_emails = [email for email in meeting_info.get("allEmails", []) if email.split("@")[0].replace(".", " ").lower() in s_lower or s_lower in email.split("@")[0].lower()]
            if any(h_domain in email for email in speaker_emails) or any(d in s_lower for d in ["sarah chen", "support agent", "engineer", "account manager"]):
                if call_type == "Support": role = "Support Agent"
                elif call_type == "Internal": role = "Engineer"
                else: role = "Account Manager"
            elif call_type == "Internal":
                role = "Engineer"
                
            connection.execute("""
            INSERT INTO transcript_segments (id, meeting_id, sentence, speaker_name, speaker_role, sentiment_type, time, end_time, average_confidence, turn_index)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            """, (
                f"{meeting_id}_t_{idx}", meeting_id, sentence, speaker, role,
                turn.get("sentimentType"), turn.get("time"), turn.get("endTime"),
                turn.get("averageConfidence"), turn.get("index")
            ))
            
            nodes_to_insert[speaker] = (speaker, 'Speaker')
            edges_to_insert.add((meeting_id, speaker, 'HAS_SPEAKER'))
            
            # Competitors detection
            s_text = sentence.lower()
            for comp in COMPETITORS:
                if re.search(rf"\b{comp.lower()}\b", s_text):
                    mentioned_competitors.add(comp)
                    edges_to_insert.add((speaker, comp, 'MENTIONS'))
                    
            # Products detection
            for prod in PRODUCTS:
                prod_short = prod.replace("Aegis ", "").lower()
                if re.search(rf"\b{prod_short}\b", s_text) or re.search(rf"\b{prod.lower()}\b", s_text):
                    mentioned_products.add(prod)
                    edges_to_insert.add((speaker, prod, 'DISCUSSES'))
                    
        # Write unique entities found
        for comp in mentioned_competitors:
            connection.execute("INSERT INTO entities VALUES (?, ?, 'Competitor') ON CONFLICT DO NOTHING", (meeting_id, comp))
            edges_to_insert.add((meeting_id, comp, 'MENTIONS_COMPETITOR'))
            
        for prod in mentioned_products:
            connection.execute("INSERT INTO entities VALUES (?, ?, 'Product') ON CONFLICT DO NOTHING", (meeting_id, prod))
            edges_to_insert.add((meeting_id, prod, 'DISCUSSES_PRODUCT'))
            
        # Connect speakers in metadata to topics
        for t in summary.get("topics", []):
            for speaker in speaker_meta.values():
                edges_to_insert.add((speaker, t, 'DISCUSSES_TOPIC'))
                
        processed += 1
        
    # Batch load graph data
    print(f"Writing {len(nodes_to_insert)} nodes and {len(edges_to_insert)} edges to Graph warehouse...")
    for nid, (label, ntype) in nodes_to_insert.items():
        connection.execute("""
        INSERT INTO graph_nodes (id, label, type) VALUES (?, ?, ?) 
        ON CONFLICT (id) DO UPDATE SET label = excluded.label, type = excluded.type
        """, (nid, label, ntype))
        
    for src, tgt, rel in edges_to_insert:
        connection.execute("INSERT INTO graph_edges (source, target, relation) VALUES (?, ?, ?) ON CONFLICT DO NOTHING", (src, tgt, rel))
        
    print(f"Pipeline completed successfully! Loaded {processed} meetings.")
    if errors:
        print(f"Encountered {len(errors)} warnings:")
        for err in errors[:5]: print(f"- {err}")

## 4. Ingesting Raw Dataset

Let's run the ingestion on our raw dataset directory located at `interview-assignment/dataset`.

In [ ]:
dataset_dir = "interview-assignment/dataset"
if not os.path.exists(dataset_dir):
    dataset_dir = "../interview-assignment/dataset"

ingest_transcripts_pipeline(dataset_dir, conn)

## 5. Ingestion Pipeline Integrity Verification

Let's run basic count queries on the tables to verify the data ingestion succeeded.

In [ ]:
tables = ['meetings', 'transcript_segments', 'action_items', 'topics', 'key_moments', 'entities', 'graph_nodes', 'graph_edges']
for t in tables:
    count = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"Table '{t}' has {count} records.")

In [ ]:
# Show a sample of Resolved Customers & Call Types
sample_meetings = conn.execute("""
    SELECT title, call_type, overall_sentiment, sentiment_score 
    FROM meetings 
    LIMIT 5
""").df()
print("Sample Ingested Meetings:")
print(sample_meetings)

In [ ]:
# Close analytical database connection
conn.close()
print("DuckDB connection closed cleanly.")